# RQ2 – Model Comparison

**Research Question:** Which supervised learning model achieves the best predictive performance for the Marketing and Product Performance Dataset, and how do ensemble methods compare to linear baselines?

**Dataset:** [Kaggle Link](https://www.kaggle.com/datasets/imranalishahh/marketing-and-product-performance-dataset)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

import glob, os

# Auto-detect the dataset file anywhere in the Colab/Kaggle environment
_search_paths = [
    '/kaggle/input/marketing-and-product-performance-dataset/',
    '/content/sample_data/',
    '/content/',
    '/content/drive/MyDrive/',
    '.',
]
_extensions = ['*.csv', '*.xlsx', '*.xls']
DATA_PATH = None

for _dir in _search_paths:
    for _ext in _extensions:
        _matches = glob.glob(os.path.join(_dir, '**', _ext), recursive=True) + glob.glob(os.path.join(_dir, _ext))
        _matches = [m for m in _matches if 'marketing' in m.lower() or 'product' in m.lower() or 'performance' in m.lower()]
        if _matches:
            DATA_PATH = _matches[0]
            break
    if DATA_PATH:
        break

# Final fallback: pick any CSV/Excel in /content
if DATA_PATH is None:
    for _ext in _extensions:
        _all = glob.glob(f'/content/**/{_ext}', recursive=True) + glob.glob(f'./**/{_ext}', recursive=True)
        if _all:
            DATA_PATH = _all[0]
            break

if DATA_PATH:
    print(f'Dataset found: {DATA_PATH}')
else:
    raise FileNotFoundError(
        'Dataset not found. Please upload the CSV/Excel file to Colab via:\n'
        '  Files panel (left sidebar) → Upload, then re-run this cell.')
RANDOM_STATE = 42
TEST_SIZE = 0.2

Dataset found: /content/marketing_and_product_performance 2.csv


In [2]:
# ── Load & Preprocess ─────────────────────────────────────────────────────────
try:
    df = pd.read_csv(DATA_PATH)
except Exception:
    df = pd.read_excel(DATA_PATH)

candidates = [c for c in df.columns if any(k in c.lower() for k in ['revenue','sales','sale','income','profit','amount'])]
TARGET = candidates[0] if candidates else df.select_dtypes(include=np.number).var().idxmax()
print(f'Target: {TARGET}')

df_clean = df.dropna(thresh=len(df)*0.5, axis=1)
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

for col in X.select_dtypes(include='object').columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X_imp = SimpleImputer(strategy='median').fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_imp)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Target: Revenue_Generated
Train: (8000, 16), Test: (2000, 16)


In [3]:
# ── Train All Models ──────────────────────────────────────────────────────────
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(random_state=RANDOM_STATE),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':           XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=RANDOM_STATE, verbosity=0),
    'SVM':               SVR(kernel='rbf', C=1.0),
    'k-NN':              KNeighborsRegressor(n_neighbors=5)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4)})
    print(f'{name:20s} | MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f}')

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
results_df

Linear Regression    | MAE=24757.9882 | RMSE=28604.0102 | R2=-0.0034
Decision Tree        | MAE=33620.2133 | RMSE=41091.0903 | R2=-1.0707
Random Forest        | MAE=25019.0251 | RMSE=28964.8929 | R2=-0.0289
XGBoost              | MAE=25042.1606 | RMSE=29166.7541 | R2=-0.0433
SVM                  | MAE=24728.5562 | RMSE=28589.7011 | R2=-0.0024
k-NN                 | MAE=26119.6576 | RMSE=31039.2160 | R2=-0.1815


,Model,MAE,RMSE,R2
0,SVM,24728.5562,28589.7011,-0.0024
1,Linear Regression,24757.9882,28604.0102,-0.0034
2,Random Forest,25019.0251,28964.8929,-0.0289
3,XGBoost,25042.1606,29166.7541,-0.0433
4,k-NN,26119.6576,31039.2160,-0.1815
5,Decision Tree,33620.2133,41091.0903,-1.0707


In [4]:
results_df.to_csv('tables/RQ2_model_comparison.csv', index=False)
print('Table saved.')

Table saved.


In [5]:
# ── Figure: Horizontal Bar Chart (R² Score) ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
palette = ['#2E4057','#048A81','#54C6EB','#F4A261','#E76F51','#6D6875']

for ax, metric in zip(axes, ['R2', 'MAE', 'RMSE']):
    sorted_df = results_df.sort_values(metric, ascending=(metric != 'R2'))
    colors = [palette[i % len(palette)] for i in range(len(sorted_df))]
    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=colors, edgecolor='white')
    ax.set_xlabel(metric, fontsize=11)
    ax.set_title(f'Model Ranking by {metric}', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(bar.get_width()*0.98, bar.get_y()+bar.get_height()/2, f'{val:.3f}',
                va='center', ha='right', fontsize=9, color='white', fontweight='bold')

fig.suptitle('Figure 2. Comparative Performance of Candidate Supervised Learning Models\n(Marketing and Product Performance Dataset)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/RQ2_model_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


## Summary

This notebook answers **RQ2** by training 6 models and comparing them across MAE, RMSE, and R². Results saved to `tables/RQ2_model_comparison.csv` and `figures/RQ2_model_comparison.pdf`.